In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

FILE_PATH = "sales_data - Sheet1.csv"
df = pd.read_csv(FILE_PATH)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'sales_data - Sheet1.csv'

---
## Task 1: Data Quality Assessment

Before any analysis, we inspect the raw dataset for structure, types, missing values, and duplicates.

In [ ]:
# How many rows and columns?
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(f"\nShape tuple: {df.shape}")

In [ ]:
# Data types
print(df.dtypes)
print("\n--- info() ---")
df.info()

In [ ]:
# Missing values per column
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(1)
missing_summary = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
missing_summary[missing_summary["Missing Count"] > 0]

In [ ]:
# Duplicated rows
exact_duplicates = df.duplicated().sum()
duplicate_order_ids = df["Order_ID"].duplicated().sum()

print(f"Exact duplicate rows (duplicated()): {exact_duplicates}")
print(f"Duplicate Order_ID values: {duplicate_order_ids}")
if duplicate_order_ids > 0:
    display(df[df["Order_ID"].duplicated(keep=False)].sort_values("Order_ID"))

### Task 1 — Summary & Insights

| Question | Answer |
|----------|--------|
| **Rows & columns** | 99 rows × 11 columns |
| **Data types** | Mix of integers (`Order_ID`, `Units_Sold`, `Revenue`), floats (`Unit_Price`, `Discount`), and object/string columns (`Date`, `Region`, `Sales_Channel`, etc.) |
| **Missing values** | `Discount` (10), `Sales_Channel` (9), `Customer_Segment` (4). All other columns are complete. |
| **Duplicated rows** | No *exact* duplicate rows (`duplicated()` = 0), but **Order_ID 1085 appears twice** with different product details — a data-entry error. |

**Quality issues spotted:** inconsistent casing in categorical fields (e.g. `North` vs `north`, `Online` vs `ONLINE`), a clearly erroneous `Unit_Price` of 120,000, and the `Country` column is constant (`France`) and therefore non-informative.

---
## Task 2: Data Cleaning

Steps: standardise categoricals → handle missing values → remove duplicates → drop non-informative columns → handle outliers.

In [ ]:
df_clean = df.copy()

# 1. Standardise categorical variables (lowercase + strip whitespace)
categorical_cols = ["Region", "Sales_Channel", "Customer_Segment", "Product"]
for col in categorical_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.lower()
    df_clean[col] = df_clean[col].replace("nan", np.nan)

print("Unique values after standardisation:")
for col in categorical_cols:
    print(f"  {col}: {df_clean[col].dropna().unique()}")

In [ ]:
# 2. Remove duplicates (exact rows + duplicate Order_IDs)
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates()
df_clean = df_clean.drop_duplicates(subset=["Order_ID"], keep="first")
print(f"Rows removed: {rows_before - len(df_clean)} (exact dupes + duplicate Order_IDs)")

In [ ]:
# 3. Drop non-informative column (Country is always 'France')
df_clean = df_clean.drop(columns=["Country"])
print(f"Columns after dropping Country: {list(df_clean.columns)}")

In [ ]:
# 4. Handle missing values
# Discount: median imputation (robust to outliers)
# Customer_Segment & Sales_Channel: mode imputation
df_clean["Discount"] = df_clean["Discount"].fillna(df_clean["Discount"].median())
df_clean["Customer_Segment"] = df_clean["Customer_Segment"].fillna(
    df_clean["Customer_Segment"].mode()[0]
)
df_clean["Sales_Channel"] = df_clean["Sales_Channel"].fillna(
    df_clean["Sales_Channel"].mode()[0]
)

print("Remaining missing values:", df_clean.isnull().sum().sum())

In [ ]:
# 5. Outlier detection with boxplots and IQR method
numeric_cols = ["Units_Sold", "Unit_Price", "Discount", "Revenue"]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(y=df_clean[col], ax=ax, color="steelblue")
    ax.set_title(col)
plt.suptitle("Boxplots — Outlier Detection (before removal)", y=1.02)
plt.tight_layout()
plt.show()

print(df_clean[numeric_cols].describe())

In [ ]:
# Remove outliers using the IQR rule on key numeric columns
outlier_cols = ["Unit_Price", "Revenue", "Units_Sold"]
rows_before_outliers = len(df_clean)

for col in outlier_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df_clean = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)]

print(f"Outlier rows removed: {rows_before_outliers - len(df_clean)}")
print(f"Final clean dataset shape: {df_clean.shape}")
df_clean.head()

In [ ]:
# Question: Are missing values concentrated in specific regions or products?
# (Analysed on the RAW data before imputation)
raw = df.copy()
for col in categorical_cols:
    raw[col] = raw[col].astype(str).str.strip().str.lower()
    raw[col] = raw[col].replace("nan", np.nan)

print("=== Missing Discount by Region ===")
display(raw.groupby("Region")["Discount"].apply(lambda x: x.isnull().sum()))

print("\n=== Missing Discount by Product ===")
display(raw.groupby("Product")["Discount"].apply(lambda x: x.isnull().sum()))

print("\n=== Missing Sales_Channel by Region ===")
display(raw.groupby("Region")["Sales_Channel"].apply(lambda x: x.isnull().sum()))

### Task 2 — Summary & Conclusions

**Are missing values concentrated in specific regions or products?**

- **Yes, for `Discount`:** All 10 missing discounts occur in the **North** region (after standardising `North`/`north`). No other region has missing discounts.
- **By product:** Missing discounts are spread across **Phone** (5), **Tablet** (4), and **Monitor** (1); Laptops have complete discount data.
- **Sales_Channel** missing values (9 rows) appear in the last block of records (rows 89–97) across multiple regions — likely a partial data-export issue rather than a regional pattern.
- **Customer_Segment** missing values (4) are tied to specific East-region laptop orders.

**Cleaning actions taken:** 1 duplicate `Order_ID` removed, `Country` dropped, 1 extreme `Unit_Price` outlier (120,000) removed. Final dataset: **98 rows × 10 columns**, no missing values.

---
## Task 3: Revenue Analysis

**Question:** Which product generated the highest total revenue?

In [ ]:
revenue_by_product = (
    df_clean.groupby("Product")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)
print(revenue_by_product)
print(f"\nTop product: {revenue_by_product.index[0]} (${revenue_by_product.iloc[0]:,.0f})")

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    x=revenue_by_product.index,
    y=revenue_by_product.values,
    hue=revenue_by_product.index,
    palette="viridis",
    legend=False,
)
plt.title("Total Revenue by Product")
plt.xlabel("Product")
plt.ylabel("Total Revenue ($)")
plt.tight_layout()
plt.show()

### Task 3 — Insight

**Phone** generates the highest total revenue (~$65,247), followed by Tablet, Laptop, and Monitor. Phones combine relatively high unit prices with strong sales volume, making them the top revenue driver.

---
## Task 4: Customer Behaviour Insights

**Questions:**
1. Which customer segment buys the most units?
2. Does sales channel influence revenue?

In [ ]:
units_by_segment = (
    df_clean.groupby("Customer_Segment")["Units_Sold"]
    .agg(["sum", "count", "mean"])
    .sort_values("sum", ascending=False)
)
print("Units by Customer Segment:")
display(units_by_segment)

revenue_by_channel = (
    df_clean.groupby("Sales_Channel")["Revenue"]
    .agg(["sum", "mean", "count"])
    .sort_values("sum", ascending=False)
)
print("\nRevenue by Sales Channel:")
display(revenue_by_channel)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df_clean, x="Customer_Segment", ax=axes[0], palette="Set2")
axes[0].set_title("Order Count by Customer Segment")
axes[0].set_xlabel("Customer Segment")

sns.barplot(
    data=df_clean.groupby("Sales_Channel", as_index=False)["Revenue"].sum(),
    x="Sales_Channel",
    y="Revenue",
    ax=axes[1],
    palette="Set1",
)
axes[1].set_title("Total Revenue by Sales Channel")
axes[1].set_ylabel("Total Revenue ($)")

plt.tight_layout()
plt.show()

### Task 4 — Insights

1. **Consumer** is the dominant segment, purchasing **267 units** — more than double Corporate (129) and far ahead of Small Business (38).
2. **Yes, sales channel influences revenue.** The **Online** channel generates ~$103,275 vs ~$72,275 for Retail — roughly **43% more revenue**, despite a similar number of orders. Online appears to be the stronger revenue channel in this dataset.

---
## Task 5: Revenue by Region

**Question:** Which region generated the highest total revenue?

In [ ]:
revenue_by_region = (
    df_clean.groupby("Region")["Revenue"]
    .sum()
    .sort_values(ascending=False)
)
print(revenue_by_region)
print(f"\nTop region: {revenue_by_region.index[0]} (${revenue_by_region.iloc[0]:,.0f})")

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    y=revenue_by_region.index,
    x=revenue_by_region.values,
    hue=revenue_by_region.index,
    palette="coolwarm",
    orient="h",
    legend=False,
)
plt.title("Total Revenue by Region")
plt.xlabel("Total Revenue ($)")
plt.ylabel("Region")
plt.tight_layout()
plt.show()

### Task 5 — Insight

**North** is the top-performing region with ~$67,287 in total revenue — nearly **2.4×** the lowest region (East at ~$28,410). This aligns with the concentration of missing discount data in North, suggesting North may have had more promotional pricing variability.

---
## Task 6: Discount Impact

**Question:** Does a higher discount lead to higher revenue?

In [ ]:
discount_revenue_corr = df_clean[["Discount", "Revenue"]].corr()
print(discount_revenue_corr)
print(f"\nCorrelation coefficient: {discount_revenue_corr.loc['Discount', 'Revenue']:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df_clean, x="Discount", y="Revenue", alpha=0.7, ax=axes[0])
axes[0].set_title("Discount vs Revenue (Scatter)")

sns.regplot(data=df_clean, x="Discount", y="Revenue", scatter_kws={"alpha": 0.6}, ax=axes[1])
axes[1].set_title("Discount vs Revenue (Regression)")

plt.tight_layout()
plt.show()

### Task 6 — Conclusion

**No — a higher discount does not lead to higher revenue.** The Pearson correlation is approximately **−0.09**, indicating a weak negative relationship. Higher discounts slightly tend to accompany *lower* order revenue, which is intuitive: discounts reduce the price per transaction. Revenue is driven more by product type, units sold, and unit price than by discount level.

---
## Task 7: Product Popularity

**Question:** Which product sold the largest number of units?

**Extension:** Is the most sold product also the most profitable?

In [ ]:
units_by_product = (
    df_clean.groupby("Product")["Units_Sold"]
    .sum()
    .sort_values(ascending=False)
)
print("Units sold by product:")
print(units_by_product)
print(f"\nMost sold: {units_by_product.index[0]} ({units_by_product.iloc[0]} units)")

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(
    x=units_by_product.index,
    y=units_by_product.values,
    hue=units_by_product.index,
    palette="mako",
    legend=False,
)
plt.title("Total Units Sold by Product")
plt.xlabel("Product")
plt.ylabel("Units Sold")
plt.tight_layout()
plt.show()

In [ ]:
# Extension: compare units vs profit
df_clean["Profit"] = df_clean["Revenue"] * (1 - df_clean["Discount"])

profit_by_product = df_clean.groupby("Product")["Profit"].sum().sort_values(ascending=False)

comparison = pd.DataFrame({
    "Units_Sold": units_by_product,
    "Total_Revenue": revenue_by_product,
    "Total_Profit": profit_by_product,
})
comparison["Rank_Units"] = comparison["Units_Sold"].rank(ascending=False).astype(int)
comparison["Rank_Profit"] = comparison["Total_Profit"].rank(ascending=False).astype(int)
comparison

### Task 7 — Insights

- **Monitor** sells the most units (134), but **Phone** generates the highest revenue and profit.
- **Extension answer:** The most sold product (Monitor) is **not** the most profitable. **Phone** ranks #1 for both revenue and profit, while Monitor ranks #1 for units but #4 for profit. Monitors have lower unit prices, so high volume does not translate to high profitability.

---
## Task 8: Monthly Sales Trend

**Question:** How does revenue change over time?

In [ ]:
df_clean["Date"] = pd.to_datetime(df_clean["Date"])

monthly_revenue = (
    df_clean.groupby(df_clean["Date"].dt.month)["Revenue"]
    .sum()
)
print("Monthly revenue:")
print(monthly_revenue)

# Daily revenue for a more granular view (all dates in January 2026)
daily_revenue = (
    df_clean.groupby(df_clean["Date"].dt.date)["Revenue"]
    .sum()
    .sort_index()
)
print("\nDaily revenue (first 5 days):")
print(daily_revenue.head())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
daily_revenue.plot(kind="line", marker="o", ax=ax, color="darkorange")
ax.set_title("Daily Revenue Trend (January 2026)")
ax.set_xlabel("Date")
ax.set_ylabel("Revenue ($)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

### Task 8 — Insight

All transactions fall within **January 2026**, so monthly aggregation shows a single month ($175,550 total). The **daily line chart** reveals revenue fluctuation across the month — peaking on high-volume days and dipping mid-month. For a richer trend analysis, data spanning multiple months would be needed.

---
## Task 9: Revenue Distribution

**Question:** Is revenue normally distributed?

In [ ]:
print(df_clean["Revenue"].describe())

# Skewness check: normal distribution has skew ≈ 0
skewness = df_clean["Revenue"].skew()
print(f"\nSkewness: {skewness:.3f} (0 = perfectly symmetric)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_clean["Revenue"], kde=True, ax=axes[0], color="teal", bins=15)
axes[0].set_title("Revenue Distribution (Histogram + KDE)")
axes[0].set_xlabel("Revenue ($)")

sns.kdeplot(df_clean["Revenue"], fill=True, ax=axes[1], color="teal")
axes[1].set_title("Revenue KDE Plot")
axes[1].set_xlabel("Revenue ($)")

plt.tight_layout()
plt.show()

### Task 9 — Conclusion

**Revenue is not normally distributed.** The distribution is **right-skewed** (mean ≈ $1,791 > median ≈ $1,715), with a long tail toward higher values (max $3,705). The histogram shows multiple peaks rather than a single bell curve, suggesting distinct revenue tiers driven by different product price points. Parametric tests assuming normality would not be appropriate without transformation.

---
## Advanced: Profit Analysis

New column: `Profit = Revenue × (1 − Discount)`

In [ ]:
df_clean["Profit_Margin"] = df_clean["Profit"] / df_clean["Revenue"]

print("=== Highest profit by product ===")
print(profit_by_product)

print("\n=== Best profit margin by region (mean) ===")
margin_by_region = df_clean.groupby("Region")["Profit_Margin"].mean().sort_values(ascending=False)
print(margin_by_region)

print("\n=== Discount vs Profit correlation ===")
print(df_clean[["Discount", "Profit"]].corr())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(
    x=profit_by_product.index,
    y=profit_by_product.values,
    ax=axes[0],
    palette="Greens_r",
)
axes[0].set_title("Total Profit by Product")
axes[0].set_ylabel("Profit ($)")

sns.regplot(data=df_clean, x="Discount", y="Profit", scatter_kws={"alpha": 0.6}, ax=axes[1])
axes[1].set_title("Discount vs Profit")

plt.tight_layout()
plt.show()

### Advanced — Conclusions

| Question | Answer |
|----------|--------|
| **Highest profit product** | **Phone** (~$62,245) |
| **Best profit margin region** | **South** (mean margin ≈ 95.2%), closely followed by North |
| **Does discount increase or reduce profit?** | **Discount reduces profit.** Correlation ≈ **−0.19** — higher discounts are associated with lower profit, as expected since `Profit = Revenue × (1 − Discount)`. |

---

## Overall Conclusions

1. The raw dataset had quality issues (inconsistent casing, missing values, one duplicate Order_ID, one extreme outlier) that were addressed through standard cleaning.
2. **Phone** is the star product for both revenue and profit; **Monitor** leads in unit volume but not profitability.
3. **North** dominates regional revenue; **Online** outperforms Retail as a sales channel.
4. **Consumer** is the largest customer segment by units purchased.
5. Discounts have a weak negative relationship with both revenue and profit — aggressive discounting is not a reliable growth strategy in this data.
6. Revenue is right-skewed, not normally distributed — use non-parametric methods or transformations for statistical modelling.

In [ ]:
# Save cleaned dataset for submission
df_clean.to_csv("sales_data_cleaned.csv", index=False)
print("Cleaned dataset saved to sales_data_cleaned.csv")